# News Summarizer Agent

Run these cells from top to bottom to fetch articles, inspect their structure, and optionally create an AI briefing.

In [ ]:
import os

print("os imported")

News agent imported.


In [ ]:
from pathlib import Path

print("Path imported")

In [ ]:
import requests

print("requests imported")

In [ ]:
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
print("dotenv loaded")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

print("message classes imported")

In [ ]:
from langchain_openai import ChatOpenAI

print("OpenAI chat model imported")

In [ ]:
DEFAULT_COUNT = 5
MAX_COUNT = 20

topic = "artificial intelligence"
count = 3
print(topic, count)

## 1. Validate a topic request

This is the first piece of logic we would later move into `agent.py`.

In [4]:
def validate_request(topic: str, count: int) -> tuple[str, int]:
    """Clean the topic and make sure the requested count is safe."""

    clean_topic = topic.strip()
    if not clean_topic:
        raise ValueError("Topic cannot be empty.")
    if not 1 <= count <= MAX_COUNT:
        raise ValueError(f"Count must be between 1 and {MAX_COUNT}.")
    return clean_topic, count


topic, count = validate_request(" artificial intelligence ", 3)
print(f"Topic: {topic}")
print(f"Article count: {count}")

NameError: name 'MAX_COUNT' is not defined

## 2. Create offline sample articles

Define one function in this cell. The next cell tests it.

In [ ]:
def build_mock_articles(topic: str) -> list[dict]:
    """Create predictable sample articles for offline exploration."""

    return [
        {"title": f"Major development in {topic}", "description": f"Researchers announce a breakthrough in {topic}.", "url": "https://example.com/1", "source": {"name": "Tech News"}},
        {"title": f"{topic.title()} industry sees rapid growth", "description": f"A new report shows {topic} adoption rising.", "url": "https://example.com/2", "source": {"name": "Business Daily"}},
        {"title": f"Experts discuss {topic} challenges", "description": f"Experts discuss challenges facing {topic}.", "url": "https://example.com/3", "source": {"name": "Science Weekly"}},
    ]

Articles found: 3
- Major development in artificial intelligence (Tech News)
- Artificial Intelligence industry sees rapid growth (Business Daily)
- Experts weigh in on artificial intelligence challenges (Science Weekly)


In [ ]:
mock_articles = build_mock_articles(topic)
print(f"Mock articles: {len(mock_articles)}")
print(mock_articles[0])

## 3. Build the briefing prompt

Define the prompt-building function separately, then inspect its output.

In [ ]:
def fetch_news(topic: str, count: int) -> list[dict]:
    """Fetch NewsAPI articles or use local mock data."""

    topic, count = validate_request(topic, count)
    news_api_key = os.getenv("NEWS_API_KEY")
    if not news_api_key:
        return build_mock_articles(topic)[:count]

    response = requests.get(
        "https://newsapi.org/v2/everything",
        params={"q": topic, "language": "en", "pageSize": count, "sortBy": "publishedAt", "apiKey": news_api_key},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    if data.get("status") != "ok":
        raise RuntimeError(data.get("message", "NewsAPI request failed."))
    return data.get("articles", [])

In [ ]:
articles = fetch_news(topic, count)
print(f"Fetched articles: {len(articles)}")

In [ ]:
def build_briefing_messages(topic: str, articles: list[dict]) -> list:
    """Create the system and user messages for the news analyst."""

    articles_text = "\n\n".join(
        f"Title: {article.get('title', 'Untitled')}\n"
        f"Source: {article.get('source', {}).get('name', 'Unknown')}\n"
        f"Summary: {article.get('description', 'N/A')}"
        for article in articles[:5]
    )
    return [
        SystemMessage(content="Create a structured news briefing with a top story, three key themes, what to watch, and quick headlines."),
        HumanMessage(content=f"Topic: {topic}\n\nArticles:\n{articles_text}"),
    ]

In [ ]:
messages = build_briefing_messages(topic, articles)
print(messages[1].content)

## 4. Summarize the news

Define the model call separately. This is the last function in the exploration.

In [ ]:
def summarize_news(topic: str, articles: list[dict]) -> str:
    """Call the language model and return a structured briefing."""

    if not articles:
        raise ValueError("No articles are available to summarize.")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm.invoke(build_briefing_messages(topic, articles))
    return response.content

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: OPENAI_API_KEY is not configured.")
else:
    print(summarize_news(topic, articles))